In [1]:
import pandas as pd

# -----------------------------
# Load data
# -----------------------------
acai = pd.read_csv("../data/ACAI-expert-annotations-10.csv")
prolific = pd.read_csv("../data/Prolific-Annotations-Pivoted.csv")

# -----------------------------
# Standardize merge key
# -----------------------------
acai = acai.rename(columns={"Index": "UID"})

# -----------------------------
# Keep only overlapping UIDs
# -----------------------------
common_uids = set(acai["UID"]).intersection(prolific["UID"])
acai = acai[acai["UID"].isin(common_uids)]
prolific = prolific[prolific["UID"].isin(common_uids)]

# -----------------------------
# Extract ACAI expert scores
# -----------------------------
dimensions = ["A1","A2","B1","B2","B3","B4","C1","C2","C3","D1","D2"]
acai_scores = acai[["UID"] + dimensions]

# -----------------------------
# Compute k-averaged Prolific scores
# -----------------------------
avg_data = {"UID": prolific["UID"]}

for dim in dimensions:
    cols = [f"k1-{dim}", f"k2-{dim}", f"k3-{dim}"]
    avg_data[dim] = prolific[cols].mean(axis=1)

prolific_avg = pd.DataFrame(avg_data)

# -----------------------------
# Merge final DataFrame
# -----------------------------
merged_df = acai_scores.merge(
    prolific_avg,
    on="UID",
    suffixes=("_expert", "_prolific")
)

import pandas as pd

# -----------------------------
# Mapping function (yours)
# -----------------------------
def map_value(answer):
    if not isinstance(answer, str):
        return None
    a = answer.lower()
    if "present" in a:
        return 1.0
    if "partial" in a:
        return 0.5
    if any(k in a for k in ["absent", "unclear", "conflicting"]):
        return 0.0
    return None

# -----------------------------
# Apply to merged_df expert cols
# -----------------------------
dimensions = ["A1","A2","B1","B2","B3","B4","C1","C2","C3","D1","D2"]

for dim in dimensions:
    merged_df[f"{dim}_expert"] = merged_df[f"{dim}_expert"].apply(map_value)

# Enforce numeric dtype
merged_df[[f"{d}_expert" for d in dimensions]] = (
    merged_df[[f"{d}_expert" for d in dimensions]].astype(float)
)

# -----------------------------
# Result
# -----------------------------
display(merged_df)



,UID,A1_expert,A2_expert,B1_expert,B2_expert,B3_expert,B4_expert,C1_expert,C2_expert,C3_expert,...,A2_prolific,B1_prolific,B2_prolific,B3_prolific,B4_prolific,C1_prolific,C2_prolific,C3_prolific,D1_prolific,D2_prolific
0,U16,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.000000,1.000000,0.666667,1.000000,1.000000,1.000000,0.500000,0.333333,0.666667,0.666667
1,U26,0.5,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.833333,0.833333,0.666667,0.666667,1.000000,0.666667,0.666667,0.000000,0.666667,0.833333
2,U18,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.666667,1.000000,0.666667,0.666667,1.000000,0.833333,0.666667,0.500000,0.833333,0.333333
3,U5,0.5,1.0,1.0,1.0,1.0,1.0,0.5,0.0,0.0,...,0.833333,1.000000,0.833333,1.000000,0.666667,0.833333,0.166667,0.500000,0.666667,0.500000
4,U35,0.5,1.0,1.0,1.0,1.0,0.0,0.5,0.0,0.0,...,1.000000,1.000000,0.666667,0.833333,0.666667,0.833333,0.333333,0.666667,0.666667,0.500000
5,U68,1.0,0.5,1.0,0.0,0.0,0.5,0.0,0.0,0.0,...,0.666667,1.000000,0.500000,0.333333,0.500000,0.333333,0.166667,0.333333,0.166667,0.500000
6,U20,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.5,...,1.000000,0.833333,0.666667,0.833333,0.833333,0.666667,0.333333,0.500000,0.000000,0.666667
7,U62,0.5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.833333,0.166667,0.500000,0.166667,0.166667,0.666667,0.166667,0.333333,0.500000,0.500000
8,U43,1.0,1.0,1.0,0.5,0.0,1.0,1.0,0.0,1.0,...,0.666667,1.000000,0.833333,0.166667,0.500000,0.166667,0.000000,0.166667,0.333333,0.500000
9,U14,0.5,0.0,1.0,0.5,1.0,1.0,1.0,0.0,1.0,...,1.000000,1.000000,0.833333,1.000000,1.000000,0.833333,0.500000,0.166667,0.333333,0.333333


In [2]:
from scipy.stats import pearsonr, spearmanr, kendalltau
import numpy as np
import pandas as pd

dimensions = ["A1","A2","B1","B2","B3","B4","C1","C2","C3","D1","D2"]

expert_all = merged_df[[f"{d}_expert" for d in dimensions]].to_numpy().ravel()
prolific_all = merged_df[[f"{d}_prolific" for d in dimensions]].to_numpy().ravel()

mask = ~pd.isna(expert_all) & ~pd.isna(prolific_all)
expert_all = expert_all[mask]
prolific_all = prolific_all[mask]

pearson = pearsonr(expert_all, prolific_all)
spearman = spearmanr(expert_all, prolific_all)
kendall = kendalltau(expert_all, prolific_all)

pearson, spearman, kendall


(PearsonRResult(statistic=np.float64(0.5551755901771168), pvalue=np.float64(3.087521916167706e-10)),
 SignificanceResult(statistic=np.float64(0.5703402530660034), pvalue=np.float64(7.809849607626705e-11)),
 SignificanceResult(statistic=np.float64(0.4928588366491839), pvalue=np.float64(1.228658444259791e-09)))

In [3]:
import numpy as np
import pandas as pd
dimensions = ["A1","A2","B1","B2","B3","B4","C1","C2","C3","D1","D2"]

expert_all = merged_df[[f"{d}_expert" for d in dimensions]].to_numpy().ravel()
prolific_all = merged_df[[f"{d}_prolific" for d in dimensions]].to_numpy().ravel()

mask = ~pd.isna(expert_all) & ~pd.isna(prolific_all)
diff = prolific_all[mask] - expert_all[mask]

diff.mean()


np.float64(0.10454545454545454)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr, kendalltau

# =====================================================
# Load data
# =====================================================
acai_expert = pd.read_csv("../data/ACAI-expert-annotations-10.csv")
prolific = pd.read_csv("../data/Prolific-Annotations-Pivoted.csv")
llm = pd.read_csv("../data/ACAI-US79-LLM-ACAI.csv")

# =====================================================
# Standardize merge keys
# =====================================================
acai_expert = acai_expert.rename(columns={"Index": "UID"})
llm = llm.rename(columns={"Index": "UID"})

dimensions = ["A1","A2","B1","B2","B3","B4","C1","C2","C3","D1","D2"]

# =====================================================
# Restrict to overlapping universities
# =====================================================
common_uids = (
    set(acai_expert["UID"])
    & set(prolific["UID"])
    & set(llm["UID"])
)

acai_expert = acai_expert[acai_expert["UID"].isin(common_uids)]
prolific = prolific[prolific["UID"].isin(common_uids)]
llm = llm[llm["UID"].isin(common_uids)]

# =====================================================
# Expert: map categorical → numeric
# =====================================================
def map_value(answer):
    if not isinstance(answer, str):
        return None
    a = answer.lower()
    if "present" in a:
        return 1.0
    if "partial" in a:
        return 0.5
    if any(k in a for k in ["absent", "unclear", "conflicting"]):
        return 0.0
    return None

expert_df = acai_expert[["UID"] + dimensions].copy()

for d in dimensions:
    expert_df[d] = expert_df[d].apply(map_value)

expert_df[dimensions] = expert_df[dimensions].astype(float)
expert_df = expert_df.add_suffix("_expert")

# =====================================================
# Prolific: average k1 / k2 / k3
# =====================================================
prolific_avg = {"UID": prolific["UID"]}

for d in dimensions:
    prolific_avg[d] = prolific[[f"k1-{d}", f"k2-{d}", f"k3-{d}"]].mean(axis=1)

prolific_df = pd.DataFrame(prolific_avg).add_suffix("_prolific")

# =====================================================
# LLM: extract scores
# (optionally filter to one temperature, e.g. tau = 1.0)
# =====================================================
llm_filtered = llm.copy()
llm_filtered = llm_filtered[llm_filtered["run_temperature"] == 1.0]

llm_scores = (
    llm_filtered[["UID"] + [f"{d}_score" for d in dimensions]]
    .rename(columns={f"{d}_score": d for d in dimensions})
    .add_suffix("_llm")
)

# =====================================================
# Merge all three
# =====================================================
merged_df = (
    expert_df
    .merge(prolific_df, left_on="UID_expert", right_on="UID_prolific")
    .merge(llm_scores, left_on="UID_expert", right_on="UID_llm")
)

merged_df = merged_df.rename(columns={"UID_expert": "UID"}).drop(
    columns=["UID_prolific", "UID_llm"]
)

display(merged_df)


# =====================================================
# Overall agreement + bias analysis
# =====================================================
def sig_stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def overall_metrics(x, y, x_label="X", y_label="Y"):
    mask = ~pd.isna(x) & ~pd.isna(y)
    x, y = x[mask], y[mask]

    pr = pearsonr(x, y)
    sr = spearmanr(x, y)
    kr = kendalltau(x, y)
    md = (y - x).mean()

    if md > 0:
        direction = f"{y_label} higher"
    elif md < 0:
        direction = f"{x_label} higher"
    else:
        direction = "equal on average"

    return {
        "pearson_r": f"{pr.statistic:.2f}{sig_stars(pr.pvalue)}",
        "spearman_rho": f"{sr.statistic:.2f}{sig_stars(sr.pvalue)}",
        "kendall_tau": f"{kr.statistic:.2f}{sig_stars(kr.pvalue)}",
        "mean_diff": round(md, 2),
        "mean_direction": direction
    }


expert_all = merged_df[[f"{d}_expert" for d in dimensions]].to_numpy().ravel()
prolific_all = merged_df[[f"{d}_prolific" for d in dimensions]].to_numpy().ravel()
llm_all = merged_df[[f"{d}_llm" for d in dimensions]].to_numpy().ravel()

results = {
    "expert_vs_prolific": overall_metrics(
        expert_all, prolific_all,
        x_label="Expert", y_label="Prolific"
    ),
    "expert_vs_llm": overall_metrics(
        expert_all, llm_all,
        x_label="Expert", y_label="LLM"
    ),
    "prolific_vs_llm": overall_metrics(
        prolific_all, llm_all,
        x_label="Prolific", y_label="LLM"
    ),
}

results_df = (
    pd.DataFrame(results)
    .T
    .reset_index()
    .rename(columns={"index": "comparison"})
)

display(results_df)



,UID,A1_expert,A2_expert,B1_expert,B2_expert,B3_expert,B4_expert,C1_expert,C2_expert,C3_expert,...,A2_llm,B1_llm,B2_llm,B3_llm,B4_llm,C1_llm,C2_llm,C3_llm,D1_llm,D2_llm
0,U16,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.500000,1.0,0.833333,0.833333,1.000000,1.000000,0.166667,0.333333,0.833333,0.333333
1,U26,0.5,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.500000,1.0,0.833333,0.833333,1.000000,0.666667,0.166667,0.166667,0.666667,0.833333
2,U18,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.500000,1.0,1.000000,1.000000,1.000000,1.000000,0.666667,0.000000,1.000000,0.500000
3,U5,0.5,1.0,1.0,1.0,1.0,1.0,0.5,0.0,0.0,...,0.500000,1.0,0.833333,0.666667,0.833333,0.666667,0.333333,0.000000,1.000000,0.000000
4,U35,0.5,1.0,1.0,1.0,1.0,0.0,0.5,0.0,0.0,...,0.333333,1.0,0.500000,0.833333,0.500000,1.000000,0.833333,0.500000,0.333333,0.333333
5,U68,1.0,0.5,1.0,0.0,0.0,0.5,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,U20,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.5,...,0.666667,1.0,1.000000,1.000000,1.000000,0.500000,0.166667,0.166667,0.500000,0.166667
7,U62,0.5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.500000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,U43,1.0,1.0,1.0,0.5,0.0,1.0,1.0,0.0,1.0,...,0.500000,1.0,0.666667,0.000000,0.500000,0.000000,0.000000,0.500000,1.000000,0.500000
9,U14,0.5,0.0,1.0,0.5,1.0,1.0,1.0,0.0,1.0,...,0.500000,1.0,0.833333,1.000000,0.500000,1.000000,0.500000,0.166667,0.000000,0.000000


,comparison,pearson_r,spearman_rho,kendall_tau,mean_diff,mean_direction
0,expert_vs_prolific,0.56***,0.57***,0.49***,0.1,Prolific higher
1,expert_vs_llm,0.60***,0.60***,0.51***,-0.01,Expert higher
2,prolific_vs_llm,0.67***,0.69***,0.57***,-0.12,Prolific higher


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr, kendalltau

# =====================================================
# Load data
# =====================================================
prolific = pd.read_csv("../data/Prolific-Annotations-Pivoted.csv")
llm = pd.read_csv("../data/ACAI-US79-LLM-ACAI.csv")

# =====================================================
# Standardize merge keys
# =====================================================
llm = llm.rename(columns={"Index": "UID"})

dimensions = ["A1","A2","B1","B2","B3","B4","C1","C2","C3","D1","D2"]

# =====================================================
# Restrict to prolific ∩ llm overlap ONLY
# =====================================================
common_uids = set(prolific["UID"]) & set(llm["UID"])
prolific = prolific[prolific["UID"].isin(common_uids)].copy()
llm = llm[llm["UID"].isin(common_uids)].copy()

# =====================================================
# Prolific: average k1 / k2 / k3
# =====================================================
prolific_avg = {"UID": prolific["UID"]}

for d in dimensions:
    prolific_avg[d] = prolific[
        [f"k1-{d}", f"k2-{d}", f"k3-{d}"]
    ].mean(axis=1)

prolific_df = pd.DataFrame(prolific_avg).add_suffix("_prolific")

# =====================================================
# LLM: extract scores (single temperature)
# =====================================================
llm_filtered = llm[llm["run_temperature"] == 1.0].copy()

llm_df = (
    llm_filtered[["UID"] + [f"{d}_score" for d in dimensions]]
    .rename(columns={f"{d}_score": d for d in dimensions})
    .add_suffix("_llm")
)

# =====================================================
# Merge prolific × llm (ALL universities)
# =====================================================
merged_pl_llm = (
    prolific_df
    .merge(llm_df, left_on="UID_prolific", right_on="UID_llm")
    .rename(columns={"UID_prolific": "UID"})
    .drop(columns=["UID_llm"])
)

display(merged_pl_llm)

# =====================================================
# Agreement + bias analysis
# =====================================================
def sig_stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def overall_metrics(x, y, x_label="X", y_label="Y"):
    mask = ~pd.isna(x) & ~pd.isna(y)
    x, y = x[mask], y[mask]

    pr = pearsonr(x, y)
    sr = spearmanr(x, y)
    kr = kendalltau(x, y)
    md = (y - x).mean()

    if md > 0:
        direction = f"{y_label} higher"
    elif md < 0:
        direction = f"{x_label} higher"
    else:
        direction = "equal on average"

    return {
        "pearson_r": f"{pr.statistic:.2f}{sig_stars(pr.pvalue)}",
        "spearman_rho": f"{sr.statistic:.2f}{sig_stars(sr.pvalue)}",
        "kendall_tau": f"{kr.statistic:.2f}{sig_stars(kr.pvalue)}",
        "mean_diff": round(md, 2),
        "mean_direction": direction,
        # "n_pairs": len(x)
    }

prolific_vals = merged_pl_llm[
    [f"{d}_prolific" for d in dimensions]
].to_numpy().ravel()

llm_vals = merged_pl_llm[
    [f"{d}_llm" for d in dimensions]
].to_numpy().ravel()

results = overall_metrics(
    prolific_vals,
    llm_vals,
    x_label="Prolific",
    y_label="LLM"
)

results_df = pd.DataFrame(
    [{"comparison": "prolific_vs_llm_all", **results}]
)

display(results_df)


,UID,A1_prolific,A2_prolific,B1_prolific,B2_prolific,B3_prolific,B4_prolific,C1_prolific,C2_prolific,C3_prolific,...,A2_llm,B1_llm,B2_llm,B3_llm,B4_llm,C1_llm,C2_llm,C3_llm,D1_llm,D2_llm
0,U1,1.000000,0.833333,1.000000,0.833333,0.833333,0.333333,1.000000,0.166667,0.333333,...,0.500000,0.833333,0.500000,0.833333,0.000000,1.000000,0.000000,0.500000,0.000000,0.000000
1,U10,0.833333,0.666667,0.666667,0.833333,1.000000,1.000000,1.000000,0.333333,0.333333,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.833333,0.666667,0.000000,0.000000
2,U11,0.666667,0.500000,1.000000,1.000000,0.833333,1.000000,1.000000,0.000000,0.500000,...,0.833333,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.666667,0.166667
3,U12,1.000000,0.666667,0.833333,0.500000,0.833333,0.500000,0.666667,0.500000,0.166667,...,0.500000,1.000000,0.833333,0.166667,1.000000,0.666667,0.166667,0.500000,0.666667,0.166667
4,U13,0.833333,1.000000,1.000000,0.500000,1.000000,1.000000,0.833333,0.333333,0.000000,...,0.500000,1.000000,0.500000,0.666667,0.000000,1.000000,0.000000,0.333333,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,U77,0.833333,1.000000,0.666667,0.500000,0.000000,0.166667,0.000000,0.000000,0.166667,...,0.500000,1.000000,0.666667,0.000000,0.500000,0.166667,0.000000,0.666667,0.500000,0.666667
75,U78,0.666667,0.666667,0.833333,0.833333,0.666667,1.000000,0.333333,0.333333,0.000000,...,0.500000,1.000000,1.000000,0.500000,1.000000,0.000000,0.000000,0.000000,0.166667,0.166667
76,U79,1.000000,0.666667,1.000000,0.666667,1.000000,0.500000,0.500000,0.666667,0.333333,...,0.333333,1.000000,0.833333,1.000000,0.500000,0.500000,0.333333,0.000000,1.000000,0.166667
77,U8,1.000000,0.833333,1.000000,1.000000,1.000000,0.333333,0.333333,0.333333,0.333333,...,0.000000,1.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000


,comparison,pearson_r,spearman_rho,kendall_tau,mean_diff,mean_direction
0,prolific_vs_llm_all,0.57***,0.57***,0.46***,-0.09,Prolific higher
